# 03 · Time-series analysis
Anomalies, trend and seasonal decomposition of the area-weighted series. Reanalysis values are model output constrained by observations — not independent observations.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline

Input: `data/processed/thetao_study_area_monthly.csv` from `scripts/build_study_area_series.py` (area-weighted over 16–34°E, 36–28°S). The box mixes the cold Benguela upwelling (west) and the warm Agulhas Current (east), so regional means blend two regimes; consider sub-regions.

In [ ]:
import pandas as pd
s = pd.read_csv('../data/processed/thetao_study_area_monthly.csv', index_col=0, parse_dates=True).iloc[:, 0]
s = s.asfreq('MS')
preprocessing.describe_series(s)

In [ ]:
clim = preprocessing.monthly_climatology(s)
fig, ax = visualisation.plot_seasonality(clim, 'Monthly climatology 1993–2025, study area')
visualisation.save(fig, '03_seasonality')

In [ ]:
anom = preprocessing.anomalies(s)
fig, ax = visualisation.plot_timeseries(anom, 'Monthly anomaly vs 1993–2025 climatology', 'Anomaly (°C)')
ax.axhline(0, color='0.3', lw=0.8)
visualisation.save(fig, '03_anomalies')

In [ ]:
from statsmodels.tsa.seasonal import STL
res = STL(s, period=12, robust=True).fit()
fig = res.plot(); fig.set_size_inches(8, 6); visualisation.save(fig, '03_stl_decomposition')

## Linear trend of anomalies
OLS with HAC (Newey–West) standard errors, because monthly anomalies are autocorrelated and naive p-values are too optimistic.

In [ ]:
import numpy as np
import statsmodels.api as sm
x = ((s.index - s.index[0]).days / 365.25).to_numpy()
ols = sm.OLS(anom.to_numpy(), sm.add_constant(x)).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
slope, se = ols.params[1] * 10, ols.bse[1] * 10
print(f'Trend: {slope:.3f} ± {1.96*se:.3f} °C/decade (95% CI, HAC), p = {ols.pvalues[1]:.3g}')
print(f'Lag-1 autocorrelation of anomalies: {anom.autocorr(1):.2f}')

In [ ]:
pd.DataFrame({'trend_degC_per_decade': [slope], 'ci95': [1.96*se], 'p_value': [ols.pvalues[1]],
              'lag1_autocorr': [anom.autocorr(1)]}).to_csv('../outputs/tables/03_trend.csv', index=False)